In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_fscore_support
import re
import math

from scipy.stats import binomtest, norm

import multiprocessing
import traceback

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

## Config

In [ ]:
# === EXPERIMENT CONFIGURATION ===

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = "/home/fahad/Documents/PROJECTS/promptmark"

#! Experiment parameters
MODELS = ["gemini", "qwen14", "codegemma", "openai", "claude"]
MODEL_NAME = MODELS[4]  #! CHANGE
EXPERIMENT_NUMBER = "exp4"  #! CHANGE
EXP_VERSION = "v1"  #! CHANGE
GENERATION_TYPE = "during"  #! 'during' or 'after'
SAMPLE_SIZE = 164   #! CHANGE
DATASET = "humaneval" #! CHANGE

# Watermark parameters
Z_THRESHOLD = 2.12 # 97.5% confidence (one-sided), equiv to ~1.73% alpha
P_THRESHOLD = norm.sf(Z_THRESHOLD)  # ✅ NOW CONSISTENT: P_THRESHOLD = norm.sf(Z_THRESHOLD) TPRX%F, X = 1.7%
GAMMA = 0.5264026402640264  # Empirically calibrated from original MBPP code corpus (for mbpp: 0.536482939632546, for humaneval: 0.5264026402640264)
SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 30  # ✅ INCREASED: n * gamma >= 5 rule of thumb (at n=30, expect 11.88 successes)
COMMENT_ENABLED = True

print(f"✅Threshold Alignment Check:")
print(f"  Z_THRESHOLD = {Z_THRESHOLD}")
print(f"  P_THRESHOLD = {P_THRESHOLD:.6f} (computed as norm.sf(Z_THRESHOLD))")
print(f"  Both represent equivalent significance level (one-sided test)")
print(f"  SMALL_SAMPLE_THRESHOLD = {SMALL_SAMPLE_THRESHOLD}")

# Derived paths
DATASET_FILE = f"human_eval_164.parquet"
DATASET_PATH = f"{BASE_DIR}/datasets/{DATASET_FILE}"
OUTPUT_DIR = f"{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}"
RESULTS_CSV = f"{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv"

In [ ]:
# # === EXPERIMENT CONFIGURATION ===

# # Project root directory (absolute path to avoid confusion with relative paths)
# BASE_DIR = "/home/fahad/Documents/PROJECTS/promptmark"

# # Watermark parameters
# Z_THRESHOLD = 2.12 
# P_THRESHOLD = norm.sf(Z_THRESHOLD)
# SEED_KEY = "exp2025"
# SMALL_SAMPLE_THRESHOLD = 30
# COMMENT_ENABLED = True

# # Derived paths
# LLM_NAME = "claude" # llama, qwen, static or gpt4
# CODE_FOLDER_NAME = f"paraphrased_codes_{LLM_NAME}"
# OUTPUT_DIR = f"{BASE_DIR}/scripts/robustness/data/{CODE_FOLDER_NAME}"
# MODEL_NAME = "gemini"
# DATASET_FILE = f"human_eval_164.parquet"
# DATASET_PATH = f"{BASE_DIR}/datasets/{DATASET_FILE}"
# RESULTS_CSV = f"{BASE_DIR}/scripts/robustness/results/{LLM_NAME}/{CODE_FOLDER_NAME}.csv"

# print(f"✅Threshold Alignment Check:")
# print(f"  Z_THRESHOLD = {Z_THRESHOLD}")
# print(f"  P_THRESHOLD = {P_THRESHOLD:.6f} (computed as norm.sf(Z_THRESHOLD))")
# print(f"  Both represent equivalent significance level (one-sided test)")
# print(f"  SMALL_SAMPLE_THRESHOLD = {SMALL_SAMPLE_THRESHOLD}")

## Helper Methods

In [ ]:
import hashlib
import random

def calculate_gamma(letter_counts: Dict[str, int], total_count: int, green_letters: set) -> float:
    if total_count == 0:
        return 0.0
    
    gamma = 0.0
    for letter in green_letters:
        if letter in letter_counts:
            p_letter = letter_counts[letter] / total_count
            gamma += p_letter
    
    return gamma

In [ ]:
## HumanEval letter frequency distribution

freq_data = {
    "letter_freqs": {
        "a": 36,
        "b": 17,
        "c": 31,
        "d": 19,
        "e": 23,
        "f": 10,
        "g": 6,
        "h": 3,
        "i": 94,
        "j": 12,
        "k": 10,
        "l": 47,
        "m": 23,
        "n": 78,
        "o": 17,
        "p": 19,
        "q": 1,
        "r": 20,
        "s": 65,
        "t": 16,
        "u": 0,
        "v": 10,
        "w": 5,
        "x": 36,
        "y": 7,
        "z": 1,
    },
    "total_identifiers": 606,
}

def get_dynamic_tier_counts(G, task_id, const=SEED_KEY, wH_range=(0.40,0.60), wM_range=(0.25,0.40)) -> Dict[str, float]:
    #! deterministic seed
    seed_val = int(hashlib.md5((const).encode()).hexdigest(), 16)
    rnd = random.Random(seed_val)  # local RNG so no global side effects

    # sample weights uniformly within ranges
    wH = rnd.uniform(*wH_range)
    wM = rnd.uniform(*wM_range)

    # ensure wL within bounds by clipping if necessary
    wL = 1.0 - wH - wM
    # if wL out of [0.10, 0.35], nudge wM to make it valid
    if wL < 0.10:
        # reduce wH or wM to lift wL
        deficit = 0.10 - wL
        # reduce wH first, then wM
        wH = max(wH_range[0], wH - deficit * 0.6)
        wM = max(wM_range[0], wM - deficit * 0.4)
        wL = 1.0 - wH - wM
    elif wL > 0.35:
        excess = wL - 0.35
        wH = min(wH_range[1], wH + excess * 0.6)
        wM = min(wM_range[1], wM + excess * 0.4)
        wL = 1.0 - wH - wM

    # integer counts
    gH = max(1, int(math.floor(G * wH)))
    gM = max(1, int(math.floor(G * wM)))
    gL = G - gH - gM
    # fix rounding issues
    if gL < 1:
        # push some from gH/gM to gL
        if gM > 1: gM -= 1; gL += 1
        else: gH -= 1; gL += 1

    return {"wH": wH, "wM": wM, "wL": wL, "gH": gH, "gM": gM, "gL": gL}

def get_red_green_sets(task_id, const=SEED_KEY, freq_dict=freq_data['letter_freqs']) -> Tuple[set, set, float]:

    task_id = str(task_id)

    # ----- Step 1: sort by frequency -----
    sorted_letters = sorted(freq_dict.keys(), key=lambda k: freq_dict[k], reverse=True)
    n = len(sorted_letters)
    high = sorted_letters[:8]
    medium = sorted_letters[8:16]
    low = sorted_letters[16:]

    # print("HIGH TIER:", high)
    # print("MEDIUM TIER:", medium)
    # print("LOW TIER:", low)
    
    #! ----- Step 2: unique seed -----
    seed_val = int(hashlib.md5((const).encode()).hexdigest(), 16)
    random.seed(seed_val)

    # print("RANDOM SEED:", seed_val)
    
    # ----- Step 3: adaptive ratio -----
    ratio = 0.5 + (random.random() - 0.5) * 0.2   # between 0.4–0.6
    green_count = int(n * ratio)

    # print(f"GREEN RATIO: {ratio:.3f}, GREEN COUNT: {green_count}/{n}")
    
    # ----- Step 4: weighted selection -----
    counts = get_dynamic_tier_counts(green_count, task_id, const="exp2025")
    num_high = counts["gH"]
    num_medium = counts["gM"]
    num_low = counts["gL"]
    # num_high = int(green_count * 0.5)
    # num_medium = int(green_count * 0.3)
    # num_low = green_count - num_high - num_medium

    # print("HIGH COUNT:", num_high)
    # print("MEDIUM COUNT:", num_medium)
    # print("LOW COUNT:", num_low)
    
    green_letters = (
        random.sample(high, min(num_high, len(high))) +
        random.sample(medium, min(num_medium, len(medium))) +
        random.sample(low, min(num_low, len(low)))
    )

    # print("INITIAL GREEN LETTERS:", green_letters)
    
    # Fill up if short (due to small tiers)
    while len(green_letters) < green_count:
        extra = random.choice(sorted_letters)
        if extra not in green_letters:
            green_letters.append(extra)
    
    # Remaining letters → red set
    red_letters = [l for l in sorted_letters if l not in green_letters]
    return set(green_letters), set(red_letters), ratio


In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import keyword

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)

class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Detect method calls like self.get_possible_moves
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                # treat as a public method
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            # treat other attributes normally
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)

def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens

def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content.strip() if content.strip() else None


#! test generated output

def extract_assertions_from_check_function(check_code: str) -> List[str]:
    """Extract all assert statements from a check function."""
    try:
        tree = ast.parse(check_code)
        assertions = []
        
        for node in ast.walk(tree):
            if isinstance(node, ast.Assert):
                assertion_code = ast.unparse(node)
                assertions.append(assertion_code)
        
        return assertions
    except:
        return []

def rename_function_to_candidate(code: str) -> str:
    """Rename the first function definition to 'candidate'."""
    try:
        tree = ast.parse(code)
        
        class FunctionRenamer(ast.NodeTransformer):
            def __init__(self):
                self.renamed = False
            
            def visit_FunctionDef(self, node):
                if not self.renamed:
                    node.name = "candidate"
                    self.renamed = True
                return node
        
        renamer = FunctionRenamer()
        tree = renamer.visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except:
        return code

def run_code_with_tests(code: str, test_imports: List[str], tests: List[str], return_dict: Dict) -> Dict:
    passed, failed = 0, 0
    return_dict["error"] = ""
    return_dict["result"] = (passed, failed)
    assertions_list = []

    try:
        code = textwrap.dedent(code)
        code = rename_function_to_candidate(code)
        env = {}

        if len(test_imports) != 0:
            for imp in test_imports:
                exec(imp, env, env)

        exec(code, env, env)

        for test_idx, test in enumerate(tests, 1):
            try:
                exec(test, env, env)
                passed += 1
                assertions_list.append({
                    "test_num": test_idx,
                    "assertion": test,
                    "status": "PASSED"
                })
            except AssertionError as e:
                failed += 1
                error_msg = str(e) if str(e) else "Assertion failed"
                print("ERROR MESSAGE: ", error_msg)
                assertions_list.append({
                    "test_num": test_idx,
                    "assertion": test,
                    "status": "FAILED",
                    "message": error_msg
                })
                return_dict["error"] += f"{test}\n{error_msg}\n"
            except Exception as e:
                failed += 1
                print("ERROR MESSAGE 2: ", error_msg)

                assertions_list.append({
                    "test_num": test_idx,
                    "assertion": test,
                    "status": "ERROR",
                    "message": str(e)
                })
                return_dict["error"] += f"{test}\n{str(e)}\n"

        return_dict["result"] = (passed, failed)
        return_dict["assertions"] = str(assertions_list)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["result"] = (0, len(tests))
        return_dict["error"] = tb
        return_dict["assertions"] = str([])

    return return_dict

def safe_exec_with_tests(code: str, test_imports: List[str], tests: List[str], timeout: float = 2) -> Dict[str, Any]:
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(target=run_code_with_tests, args=(code, test_imports, tests, return_dict))
    
    p.start()
    p.join(timeout)
    
    if p.is_alive():
        p.terminate()
        return_dict['result'] = (0, len(tests))
        return_dict['error'] = "Timeout: Code execution exceeded time limit"
        return_dict['assertions'] = str([])
        return dict(return_dict)
    
    result_dict = dict(return_dict)
    
    if 'assertions' in result_dict:
        try:
            result_dict['assertions'] = eval(result_dict['assertions'])
        except:
            result_dict['assertions'] = []
    
    return result_dict

def test_code(code: str, test_imports: List[str], tests: List[str], timeout: float = 2) -> Dict:
    return safe_exec_with_tests(code, test_imports, tests, timeout)


#! Extract starting letters from comments
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []

    # create a deepcopy
    import copy

    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split("\n")

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find("#")
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1 :].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = (
                    "inline_comment" if code_before_hash else "full_line_comment"
                )

                comments.append(
                    {
                        "line_number": line_number,
                        "content": comment_content,
                        "type": comment_type,
                        "code_context": (
                            code_before_hash[:50] + "..."
                            if len(code_before_hash) > 50
                            else code_before_hash
                        ),
                    }
                )
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments

def get_comment_starting_letters(source_code: str, gamma) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        # print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment["content"].split("\n")

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r"\b[a-zA-Z]+\b", line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - gamma) / (
                (gamma * (1 - gamma)) ** 0.5 / token_count**0.5
            )
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback

        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0


#! fix the method name with the name mentioned in assert statement

def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]

def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None

def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""

    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)

def fix_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code

In [ ]:
import math
from scipy.stats import binomtest, norm


def calculate_z_score(token_count, green_count, gamma):
    """Calculate z-score for green token count (standard normal approximation)."""
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )


def calculate_p_value_exact(green_count, token_count, gamma):
    """
    Calculate exact binomial p-value (one-sided test: H1: p > gamma).
    Use this for ALL samples; it's exact and always valid.
    """
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative="greater").pvalue


def calculate_p_value_normal(z_score):
    """
    Calculate approximate p-value using standard normal CDF (one-sided).
    Used only for reference/large samples where normal approx is valid.
    """
    if math.isnan(z_score):
        return 1.0
    return norm.sf(z_score)


def get_unified_detection_score(token_count, green_count, gamma, 
                                small_threshold=SMALL_SAMPLE_THRESHOLD):
    """
    ✅ UNIFIED APPROACH: Compute both z and p_exact, then select based on sample size.
    Returns: (p_unified, z_score, p_exact, method_used)
    
    For small samples: Use exact binomial p-value (most accurate)
    For large samples: Can use either (both converge), we use p_exact for consistency
    
    Score for ranking: -log10(p_unified) ensures higher scores = more evidence of watermark
    """
    z_score = calculate_z_score(token_count, green_count, gamma)
    p_exact = calculate_p_value_exact(green_count, token_count, gamma)
    p_normal = calculate_p_value_normal(z_score)
    
    # Select which p-value to use
    if token_count < small_threshold:
        p_unified = p_exact
        method = "binomial"
    else:
        p_unified = z_score  # ✅ Use exact for consistency; normal approx also valid
        method = "exact_large"
    
    # Score for ROC: higher = more evidence. Use -log10 to spread values
    score = -np.log10(np.clip(p_unified, 1e-300, 1.0))
    
    return {
        "p_unified": p_unified,
        "z_score": z_score,
        "p_exact": p_exact,
        "p_normal": p_normal,
        "score": score,
        "method_used": method,
        "token_count": token_count,
    }


## Detection

In [ ]:
import os 

def detect_watermark(task_id, original_code, generated_code, green_letters, red_letters, gamma):
    """
    UNIFIED WATERMARK DETECTION
    - Uses consistent p-value based approach for both decisions and ROC scoring
    - Computes both exact binomial p-value and z-score
    - Decision rule: p_unified < P_THRESHOLD (one-sided)
    - Score for ROC: -log10(p_unified)
    """
    import ast

    def get_starting_letters(code):
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return [], [], 0, float("nan"), float("nan"), {}

        visitor = CodeNavigator()
        visitor.visit(tree)

        # Collect non-public identifiers (user-defined variables, functions, etc.)
        non_public_tokens = (
            visitor.non_public_classes
            | visitor.non_public_funcs
            | visitor.non_public_vars
        )

        # Get starting letters, lowercase, excluding common ones like 'self', 'cls'
        relevant_tokens = [
            token for token in non_public_tokens if token not in {"self", "cls"}
        ]

        def get_starting_letter(word):
            if not word:
                return None
            if word[0] == "_":
                if len(word) > 1 and word[1].isalpha():
                    return word[1].lower()
                else:
                    return None
            elif word[0].isalpha():
                return word[0].lower()
            else:
                return None

        starting_letters = [
            letter
            for word in relevant_tokens
            if (letter := get_starting_letter(word)) is not None
        ]

        green_count = sum(1 for letter in starting_letters if letter in green_letters)

        # Include comment starting letters if enabled
        if COMMENT_ENABLED:
            cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, _, _ = (
                get_comment_starting_letters(code, gamma)
            )
            # print(f"✅ COMMENT STARTING LETTERS: {cmnt_starting_letters}")
            print(
                f"   Comment green count: {cmnt_green_count}, Comment token count: {len(cmnt_starting_letters)}"
            )
            starting_letters.extend(cmnt_starting_letters)
            relevant_tokens.extend(cmn_relevant_words)
            green_count += cmnt_green_count

        token_count = len(starting_letters)

        # ✅ Get unified detection stats
        unified_stats = get_unified_detection_score(token_count, green_count, gamma)

        return starting_letters, relevant_tokens, green_count, unified_stats

    orig_starts, orig_relevant_tokens, orig_green_count, orig_stats = (
        get_starting_letters(original_code)
    )
    
    gen_starts, gen_relevant_tokens, gen_green_count, gen_stats = get_starting_letters(
        generated_code
    )

    #! save the starting tokens in a separate files
    os.makedirs(f"starting_tokens_gen", exist_ok=True)
    # os.makedirs(f"starting_tokens_org/{task_id}.txt", exist_ok=True)

    with open(f"starting_tokens_gen/{task_id}.py", "w") as f:
        f.write("'''")
        f.write("\n# Starting Tokens:\n")
        for token in gen_starts:
            f.write(f"{token}\n")

        f.write("\n# Relevant Tokens:\n")
        for token in gen_relevant_tokens:
            f.write(f"{token}\n")

        f.write(f"\n Stats: \n {gen_stats}")

        f.write("\n'''\n\n")

        f.write(generated_code)

    preferred = green_letters
    avoided = red_letters

    # Calculate ratios
    orig_preferred_ratio = (
        sum(1 for letter in orig_starts if letter in preferred) / len(orig_starts)
        if orig_starts
        else 0
    )
    gen_preferred_ratio = (
        sum(1 for letter in gen_starts if letter in preferred) / len(gen_starts)
        if gen_starts
        else 0
    )

    orig_avoided_count = sum(1 for letter in orig_starts if letter in avoided)
    gen_avoided_count = sum(1 for letter in gen_starts if letter in avoided)

    # ✅ UNIFIED DETECTION: Use p_unified for both decision and all returned stats
    def is_watermarked_unified(p_unified, threshold=P_THRESHOLD):
        """Simple threshold-based decision using unified p-value."""
        if math.isnan(p_unified):
            return False
        return p_unified < threshold

    return {
        # Original code stats
        "original_z_score": orig_stats["z_score"],
        "original_p_exact": orig_stats["p_exact"],
        "original_p_unified": orig_stats["p_unified"],
        "original_score": orig_stats["score"],
        "original_method_used": orig_stats["method_used"],
        "original_preferred_ratio": orig_preferred_ratio,
        "original_token_count": len(orig_relevant_tokens),
        "original_green_count": orig_green_count,
        "original_is_watermarked": is_watermarked_unified(orig_stats["p_unified"]),
        "original_unique_starts": sorted(set(orig_starts)),
        # Generated code stats
        "generated_z_score": gen_stats["z_score"],
        "generated_p_exact": gen_stats["p_exact"],
        "generated_p_unified": gen_stats["p_unified"],
        "generated_score": gen_stats["score"],
        "generated_method_used": gen_stats["method_used"],
        "generated_preferred_ratio": gen_preferred_ratio,
        "generated_token_count": len(gen_relevant_tokens),
        "generated_green_count": gen_green_count,
        "generated_is_watermarked": is_watermarked_unified(gen_stats["p_unified"]),
        "generated_unique_starts": sorted(set(gen_starts)),
    }

In [ ]:
def analyze_detection_by_sample_size(results_df):
    """
    ✅ STRATIFIED ANALYSIS: Compute detection metrics separately for small-n and large-n samples.
    This reveals whether low TPR is due to inherent small-sample limitation.
    """
    print("\n" + "="*80)
    print("📊 STRATIFIED DETECTION ANALYSIS (by sample size)")
    print("="*80)
    
    # Bin by generated code token count
    small_n_mask = results_df["generated_token_count"] < SMALL_SAMPLE_THRESHOLD
    large_n_mask = results_df["generated_token_count"] >= SMALL_SAMPLE_THRESHOLD
    
    small_n_count = small_n_mask.sum()
    large_n_count = large_n_mask.sum()
    
    print(f"\nSample Distribution:")
    print(f"  Small-n (< {SMALL_SAMPLE_THRESHOLD}): {small_n_count} samples ({100*small_n_count/len(results_df):.1f}%)")
    print(f"  Large-n (>= {SMALL_SAMPLE_THRESHOLD}): {large_n_count} samples ({100*large_n_count/len(results_df):.1f}%)")
    
    metrics_by_regime = {}
    
    for regime_name, mask in [("Small-n", small_n_mask), ("Large-n", large_n_mask)]:
        if mask.sum() == 0:
            print(f"\n⚠️  No samples in {regime_name} regime")
            continue
            
        subset_df = results_df[mask]
        
        # Build predictions and ground truth for this regime
        orig_pred = subset_df["original_is_watermarked"].tolist()
        gen_pred = subset_df["generated_is_watermarked"].tolist()
        
        y_true_regime = [0] * len(orig_pred) + [1] * len(gen_pred)
        y_pred_regime = orig_pred + gen_pred
        y_score_regime = (
            subset_df["original_score"].fillna(0).tolist() + 
            subset_df["generated_score"].fillna(0).tolist()
        )
        
        y_true_regime = np.array(y_true_regime)
        y_pred_regime = np.array(y_pred_regime)
        
        tp_r = ((y_true_regime == 1) & (y_pred_regime == 1)).sum()
        fp_r = ((y_true_regime == 0) & (y_pred_regime == 1)).sum()
        tn_r = ((y_true_regime == 0) & (y_pred_regime == 0)).sum()
        fn_r = ((y_true_regime == 1) & (y_pred_regime == 0)).sum()
        
        tpr_r = tp_r / (tp_r + fn_r) if (tp_r + fn_r) > 0 else 0
        fpr_r = fp_r / (fp_r + tn_r) if (fp_r + tn_r) > 0 else 0
        fnr_r = fn_r / (fn_r + tp_r) if (fn_r + tp_r) > 0 else 0
        tnr_r = tn_r / (tn_r + fp_r) if (tn_r + fp_r) > 0 else 0
        
        precision_r = tp_r / (tp_r + fp_r) if (tp_r + fp_r) > 0 else 0
        
        # ROC AUC for this regime
        if len(np.unique(y_true_regime)) > 1:
            fpr_r_roc, tpr_r_roc, _ = roc_curve(y_true_regime, y_score_regime)
            auc_r = auc(fpr_r_roc, tpr_r_roc)
        else:
            auc_r = None
        
        metrics_by_regime[regime_name] = {
            "sample_count": len(subset_df),
            "tp": int(tp_r), "fp": int(fp_r), "tn": int(tn_r), "fn": int(fn_r),
            "tpr": round(tpr_r, 4),
            "fpr": round(fpr_r, 4),
            "tnr": round(tnr_r, 4),
            "fnr": round(fnr_r, 4),
            "precision": round(precision_r, 4),
            "auc": round(auc_r, 4) if auc_r else None,
            "avg_generated_token_count": round(subset_df["generated_token_count"].mean(), 1),
            "avg_generated_green_count": round(subset_df["generated_green_count"].mean(), 1),
        }
        
        print(f"\n{regime_name} Regime ({metrics_by_regime[regime_name]['sample_count']} samples):")
        print(f"  Avg identifier count: {metrics_by_regime[regime_name]['avg_generated_token_count']}")
        print(f"  Avg green count: {metrics_by_regime[regime_name]['avg_generated_green_count']}")
        print(f"  TP={tp_r}, FP={fp_r}, TN={tn_r}, FN={fn_r}")
        print(f"  TPR={metrics_by_regime[regime_name]['tpr']:.4f}, FPR={metrics_by_regime[regime_name]['fpr']:.4f}")
        print(f"  Precision={metrics_by_regime[regime_name]['precision']:.4f}, AUC={metrics_by_regime[regime_name]['auc']}")
    
    return metrics_by_regime


def analyze_detection_method_usage(results_df):
    """
    ✅ Show which detection method (binomial vs exact_large) was used for each regime.
    """
    print("\n" + "="*80)
    print("🔬 DETECTION METHOD USAGE")
    print("="*80)
    
    orig_methods = results_df["original_method_used"].value_counts()
    gen_methods = results_df["generated_method_used"].value_counts()
    
    print("\nOriginal Code Detection Methods:")
    for method, count in orig_methods.items():
        print(f"  {method}: {count} ({100*count/len(results_df):.1f}%)")
    
    print("\nGenerated Code Detection Methods:")
    for method, count in gen_methods.items():
        print(f"  {method}: {count} ({100*count/len(results_df):.1f}%)")


## Load Data

In [ ]:
missing_ids = [
    'HumanEval/2', 'HumanEval/6', 'HumanEval/10','HumanEval/16', 'HumanEval/17', 'HumanEval/18', 'HumanEval/19', 'HumanEval/20', 'HumanEval/22', 'HumanEval/23', 'HumanEval/24', 'HumanEval/25', 'HumanEval/26', 'HumanEval/27', 'HumanEval/28', 'HumanEval/29', 'HumanEval/30', 'HumanEval/31', 'HumanEval/33', 'HumanEval/37', 'HumanEval/38', 'HumanEval/39', 'HumanEval/40', 'HumanEval/41', 'HumanEval/42', 'HumanEval/43', 'HumanEval/44', 'HumanEval/45', 'HumanEval/46', 'HumanEval/47', 'HumanEval/48', 'HumanEval/49', 'HumanEval/50', 'HumanEval/51', 'HumanEval/52', 'HumanEval/53', 'HumanEval/54', 'HumanEval/55', 'HumanEval/56', 'HumanEval/57', 'HumanEval/58', 'HumanEval/59', 'HumanEval/60', 'HumanEval/61', 'HumanEval/62', 'HumanEval/64', 'HumanEval/65', 'HumanEval/66', 'HumanEval/67', 'HumanEval/69', 'HumanEval/70', 'HumanEval/71', 'HumanEval/72', 'HumanEval/73', 'HumanEval/75', 'HumanEval/76', 'HumanEval/77', 'HumanEval/78', 'HumanEval/79', 'HumanEval/82', 'HumanEval/83', 'HumanEval/84', 'HumanEval/86', 'HumanEval/87', 'HumanEval/91', 'HumanEval/92', 'HumanEval/93', 'HumanEval/94', 'HumanEval/95', 'HumanEval/96', 'HumanEval/97', 'HumanEval/99', 'HumanEval/100', 'HumanEval/101', 'HumanEval/102', 'HumanEval/103', 'HumanEval/104', 'HumanEval/105', 'HumanEval/106', 'HumanEval/107', 'HumanEval/108', 'HumanEval/109', 'HumanEval/110', 'HumanEval/115', 'HumanEval/116', 'HumanEval/117', 'HumanEval/118', 'HumanEval/119', 'HumanEval/120', 'HumanEval/121', 'HumanEval/122', 'HumanEval/123', 'HumanEval/124', 'HumanEval/125', 'HumanEval/127', 'HumanEval/128', 'HumanEval/129', 'HumanEval/130', 'HumanEval/132', 'HumanEval/133', 'HumanEval/134', 'HumanEval/135', 'HumanEval/136', 'HumanEval/137', 'HumanEval/138', 'HumanEval/140', 'HumanEval/141', 'HumanEval/143', 'HumanEval/144', 'HumanEval/145', 'HumanEval/146', 'HumanEval/147', 'HumanEval/149', 'HumanEval/150', 'HumanEval/151', 'HumanEval/152', 'HumanEval/153', 'HumanEval/154', 'HumanEval/155', 'HumanEval/156', 'HumanEval/157', 'HumanEval/158', 'HumanEval/159', 'HumanEval/160', 'HumanEval/161', 'HumanEval/162', 'HumanEval/163'
]

In [ ]:
# Load dataset
print("📂 Loading dataset...")
df = pd.read_parquet(DATASET_PATH)

#! Check if results CSV exists
if os.path.exists(RESULTS_CSV):
    print(f"📂 Loading existing results from {RESULTS_CSV}")
    results_df = pd.read_csv(RESULTS_CSV)
    print(f"Results loaded: {len(results_df)} samples")
else:
    print(f"⚠️ Results CSV not found at {RESULTS_CSV}")
    print("Will analyze generated code files directly...")
    results_df = None

#! Verify output directory
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    generated_files = list(output_path.glob("*.py"))
    print(f"📂 Found {len(generated_files)} generated code files")
else:
    print(f"❌ Output directory not found: {OUTPUT_DIR}")
    generated_files = []

## Run Detection

In [ ]:
import textwrap
missing_ids = []
if results_df is None and generated_files:
    print("🔄 Analyzing generated code files...")
    analysis_results = []
    failed_cases = []

    for idx, row in df.iterrows():
        task_id = row["task_id"].split('/')[1]
        original_code = row["canonical_solution"]

        print(f"\nTask: {task_id}")

        #! Get green/red letter sets for this task
        letter_freqs = freq_data['letter_freqs']
        total_identifiers = freq_data['total_identifiers']
        global green_letters, red_letters

        green_letters, red_letters, ratio = get_red_green_sets(task_id)
        GAMMA = calculate_gamma(letter_freqs, total_identifiers, green_letters)

        generated_file = Path(OUTPUT_DIR) / f"{task_id}.py"
        if not generated_file.exists():
            missing_ids.append(row["task_id"]) 
            print(f"Missing file for {task_id}")
            continue

        generated_code = generated_file.read_text(encoding="utf-8")

        try:
            original_code = textwrap.dedent(original_code)
            watermark_result = detect_watermark(task_id, original_code, generated_code, green_letters, red_letters, GAMMA)
        except Exception as e:
            print("ERROR ", e)
            continue

        print("WM RESULT: ", watermark_result)

        test_str = row.get("test", [])
        assertions = extract_assertions_from_check_function(test_str)
        print("CODE >> ", generated_code)
        print("ASSERTIONS >> ", assertions)
        result = test_code(generated_code, [], assertions)

        passed, failed = result["result"]
        error_message = result["error"]
        total = passed + failed

        print(f"PASSED: {passed}, FAILED: {failed}")

        if failed > 0:
            failed_cases.append(task_id)
            print(f"💥 Failed tests for {task_id}: {failed}")
            print(f"Error Message\n: {error_message}")

        result = {
            "task_id": task_id,
            "tests_passed": passed,
            "tests_failed": failed,
            "total_tests": total,
            "pass_rate": round((passed / total * 100) if total > 0 else 0, 2),
            "all_passed": (passed == total),
            **watermark_result,
        }
        analysis_results.append(result)

        print(f"Processed sample {idx+1}/{len(df)}")


    print(">>>", analysis_results)

    total_passed = sum([1 for item in analysis_results if item["all_passed"]])
    total_failed = sum([1 for item in analysis_results if not item["all_passed"]])

    print(
        f"\n\n⏰ Total Passed: {total_passed}, Total Failed: {total_failed}\nFailed IDs: {failed_cases}\n\n"
    )
    print(f"Pass rate: {total_passed / (total_passed + total_failed) * 100:.2f}%")

    # Create results DataFrame
    results_df = pd.DataFrame(analysis_results)

    # Save results
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"💾 Results saved to {RESULTS_CSV}")

print(
    f"📊 Analysis complete: {len(results_df) if results_df is not None else 0} samples processed"
)

In [ ]:
print(missing_ids)
print(len(missing_ids))

In [ ]:
if results_df is not None:
    print("="*80)
    print("=== ✅ UNIFIED WATERMARK DETECTION PERFORMANCE ===")
    print("="*80)
    print("\n📌 Key Points:")
    print("   • Decisions and ROC use SAME unified p-value statistic")
    print("   • Small samples (n < %d) use exact binomial p-value" % SMALL_SAMPLE_THRESHOLD)
    print("   • Large samples (n >= %d) also use exact p-value for consistency" % SMALL_SAMPLE_THRESHOLD)
    print("   • Threshold: P_THRESHOLD = %.6f (equivalent to Z_THRESHOLD = %.3f)" % (P_THRESHOLD, Z_THRESHOLD))
    print("   • Score for ranking: -log10(p_unified) [higher = more evidence]\n")

    # ✅ Extract unified predictions and scores
    original_is_watermarked = results_df["original_is_watermarked"].tolist()
    generated_is_watermarked = results_df["generated_is_watermarked"].tolist()

    # Ground Truth: Original code should be non-watermarked (0), Generated should be watermarked (1)
    true_labels = [0] * len(original_is_watermarked) + [1] * len(generated_is_watermarked)

    # Predictions: Based on unified detection logic (p_unified < P_THRESHOLD)
    all_predictions = original_is_watermarked + generated_is_watermarked

    # ✅ Scores for ROC calculation: Use unified scores (-log10(p_unified))
    original_scores = results_df["original_score"].fillna(0).tolist()
    generated_scores = results_df["generated_score"].fillna(0).tolist()
    all_scores = original_scores + generated_scores

    # Convert to numpy arrays
    y_true = np.array(true_labels)
    y_pred = np.array(all_predictions)
    y_score = np.array(all_scores)

    # Confusion matrix components
    tp = ((y_true == 1) & (y_pred == 1)).sum()  # Generated correctly detected as watermarked
    fp = ((y_true == 0) & (y_pred == 1)).sum()  # Original incorrectly detected as watermarked
    tn = ((y_true == 0) & (y_pred == 0)).sum()  # Original correctly detected as non-watermarked
    fn = ((y_true == 1) & (y_pred == 0)).sum()  # Generated incorrectly detected as non-watermarked

    print("📊 CONFUSION MATRIX:")
    print(f"  TP (Generated → Watermarked): {tp}")
    print(f"  FP (Original → Watermarked):  {fp}")
    print(f"  TN (Original → Non-WM):       {tn}")
    print(f"  FN (Generated → Non-WM):      {fn}")
    print(f"  TOTAL:                         {tp + fp + tn + fn}")

    # Calculate metrics
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # Sensitivity/Recall
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0  # Specificity
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    f1 = 2 * (precision * tpr) / (precision + tpr) if (precision + tpr) > 0 else 0

    # ✅ ROC curve using unified scores
    fpr_roc, tpr_roc, thresholds = roc_curve(true_labels, all_scores, pos_label=1)
    roc_auc = auc(fpr_roc, tpr_roc)

    print("FPR VALUES FOR ROC CURVE:", fpr_roc)
    print("TPR VALUES FOR ROC CURVE:", tpr_roc)

    print("\n📈 PERFORMANCE METRICS:")
    print(f"  TPR (Sensitivity):  {tpr:.4f}")
    print(f"  FPR (False +ve):    {fpr:.4f}")
    print(f"  TNR (Specificity):  {tnr:.4f}")
    print(f"  FNR:                {fnr:.4f}")
    print(f"  Precision:          {precision:.4f}")
    print(f"  Accuracy:           {accuracy:.4f}")
    print(f"  F1 Score:           {f1:.4f}")
    print(f"  AUROC:              {roc_auc:.4f}")


In [ ]:
import json
if results_df is not None:
    print("\n" + "="*80)
    print("🎯 TPR AT SPECIFIC FPR THRESHOLDS (T@X%F)")
    print("="*80)
    
    def get_tpr_at_fpr(fpr_curve, tpr_curve, fpr_threshold):
        """Get TPR at specific FPR threshold from ROC curve."""
        valid_indices = np.where(fpr_curve <= fpr_threshold)[0]
        
        if len(valid_indices) == 0:
            return 0.0
        
        best_idx = valid_indices[-1]
        return float(tpr_curve[best_idx])
    
    fpr_thresholds = [0.0, 0.01, 0.03, 0.05, 0.10]
    tpr_at_fpr_values = {}
    
    print(f"\nOperating Points from ROC Curve:")
    print(f"{'FPR Threshold':<15} {'TPR Value':<15} {'Metric':<20}")
    print("-" * 50)
    
    for fpr_th in fpr_thresholds:
        tpr_val = get_tpr_at_fpr(fpr_roc, tpr_roc, fpr_th)
        metric_name = f"T@{int(fpr_th*100)}%F"
        tpr_at_fpr_values[metric_name] = round(tpr_val, 6)
        
        print(f"{fpr_th:<15.4f} {tpr_val:<15.6f} {metric_name:<20}")
    
    print(f"\n📊 AUROC: {roc_auc:.6f}")
    print(f"\n✅ T@X%F Metrics Summary:")
    for metric_name, tpr_val in tpr_at_fpr_values.items():
        print(f"   {metric_name}: {tpr_val:.6f}")
    
    tpr_fpr_results = {
        "experiment_info": {
            "z_threshold": float(Z_THRESHOLD),
            "p_threshold": float(P_THRESHOLD),
            "model": MODEL_NAME,
            "total_samples": len(results_df),
        },
        "roc_metrics": {
            "auroc": round(float(roc_auc), 6),
            "fpr_range": [float(fpr_roc.min()), float(fpr_roc.max())],
            "tpr_range": [float(tpr_roc.min()), float(tpr_roc.max())],
        },
        "tpr_at_fpr_thresholds": tpr_at_fpr_values,
        "confusion_matrix": {
            "tp": int(tp),
            "fp": int(fp),
            "tn": int(tn),
            "fn": int(fn),
        },
        "performance_metrics": {
            "tpr_at_fixed_threshold": round(float(tpr), 6),
            "fpr_at_fixed_threshold": round(float(fpr), 6),
            "tnr_specificity": round(float(tnr), 6),
            "fnr": round(float(fnr), 6),
            "precision": round(float(precision), 6),
            "accuracy": round(float(accuracy), 6),
            "f1_score": round(float(f1), 6),
        },
    }
    
    tpr_fpr_json_path = RESULTS_CSV.replace(".csv", "_tpr_at_fpr.json")
    with open(tpr_fpr_json_path, "w") as f:
        json.dump(tpr_fpr_results, f, indent=2)
    
    print(f"\n💾 TPR@X%F results saved to: {tpr_fpr_json_path}")
    print("\n📋 JSON Structure:")
    print(json.dumps(tpr_fpr_results, indent=2))

In [ ]:
if results_df is not None:
    print("\n" + "="*80)
    print("💾 SAVING RESULTS")
    print("="*80)
    
    # Create comprehensive summary metrics dictionary
    summary_metrics = {
        "experiment_info": {
            "dataset_size": len(results_df),
            "detection_method": "unified (exact binomial p-value)",
            "date": pd.Timestamp.now().isoformat(),
        },
        "thresholds": {
            "z_threshold": Z_THRESHOLD,
            "p_threshold": float(P_THRESHOLD),
            "p_threshold_note": "Computed as norm.sf(Z_THRESHOLD) for consistency",
            "small_sample_threshold": SMALL_SAMPLE_THRESHOLD,
            "gamma": GAMMA,
            "gamma_note": "Should be empirically calibrated from original code corpus",
        },
        "confusion_matrix": {
            "tp": int(tp),
            "fp": int(fp),
            "tn": int(tn),
            "fn": int(fn),
        },
        "performance_metrics": {
            "tpr_sensitivity": round(float(tpr), 4),
            "fpr_false_positive_rate": round(float(fpr), 4),
            "tnr_specificity": round(float(tnr), 4),
            "fnr_false_negative_rate": round(float(fnr), 4),
            "precision": round(float(precision), 4),
            "accuracy": round(float(accuracy), 4),
            "f1_score": round(float(f1), 4),
            "auroc": round(float(roc_auc), 4) if roc_auc is not None else None,
        },
    }

    # Add pass rate metrics if available
    if "pass_rate" in results_df.columns:
        summary_metrics["code_quality"] = {
            "avg_pass_rate": round(float(results_df["pass_rate"].mean()), 2),
        }

    # Save summary metrics
    summary_path = RESULTS_CSV.replace(".csv", "_summary_unified.json")
    import json

    with open(summary_path, "w") as f:
        json.dump(summary_metrics, f, indent=2)
    
    print(f"✅ Summary metrics saved to: {summary_path}")
    
    # Also save detailed results with new columns
    print(f"✅ Detailed results in: {RESULTS_CSV}")
    print(f"   New columns: original_p_unified, original_score, original_method_used,")
    print(f"               generated_p_unified, generated_score, generated_method_used")
    
    # Print summary to console
    print("\n📋 SUMMARY SAVED:")
    print(json.dumps(summary_metrics, indent=2))
